(pnf)=
# Parametric Normal Form

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

Everything that `twiss` computes can also be computed as a function of parameters of the
lattice: for example how the tunes depend on magnet strengths, or how the beta functions depend
on a length. These dependences are computed exactly, by automatic differentiation, and not by
finite differences.

There are two ways to do this. The simplest is to write the quantities and the parameters as
expressions, and let SciBmad compute their derivatives. The second is to make the parameters
Taylor series yourself, with the [GTPSA](https://github.com/bmad-sim/GTPSA.jl) package, and get
the complete dependence on them.

## Derivatives with respect to lattice parameters

As an example, take a FODO cell with sextupoles:

In [2]:
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    sf = sb.Sextupole(Kn2=1.2, L=0.2)
    d = sb.Drift(L=0.1)
    b = sb.SBend(L=1.2, angle=np.pi/132)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)
    sd = sb.Sextupole(Kn2=-1.2, L=0.2)

fodo = sb.Beamline([qf, sf, b, d, qd, sd, b, d],
                   species_ref=sb.Species("electron"), pc_ref=18e9)

We first construct a `TwissExpr`, which is basically a lazily-evaluated `twiss` call. It takes the
same arguments as `twiss`, except `at`, `cols`, `base_cols`, `as_taylor_series` and
`GTPSA_descriptor`, which are determined automatically given what you request. `TwissExpr`
quantities are read identically as the summary values and columns of a `twiss` result are read(see
[Twiss](twiss.md)), but nothing is computed yet. The parameters are given as knobs, e.g.
{jl}`@knob(qf.Kn1)`{py}`sb.Knob(qf, "Kn1")` for the strength of `qf`:

In [3]:
tw = sb.TwissExpr(fodo)
kf = sb.Knob(qf, "Kn1")
kd = sb.Knob(qd, "Kn1")

{jl}`SciBmad.value_and_jacobian`{py}`sb.value_and_jacobian` computes the values of the quantities
and their derivatives with
respect to the knobs. It is not exported, because `DifferentiationInterface` exports a function of
the same name. The Jacobian has one row per quantity and one column per knob:

In [4]:
v, J = sb.value_and_jacobian([tw.q1, tw.q2], [kf, kd])
J  # [[dq1/dkf, dq1/dkd], [dq2/dkf, dq2/dkd]]

array([[ 0.55396906,  0.40916228],
       [-0.41628098, -0.56355469]])

The same works for any column, at any location. For example, how the beta function and the
dispersion at `qd` depend on the quadrupole strengths:

In [5]:
v, J = sb.value_and_jacobian([tw.beta1[qd], tw.dx[qd]], [kf, kd])
J  # [[dbeta1/dkf, dbeta1/dkd], [ddx/dkf, ddx/dkd]]

array([[-110.1325315 ,  -78.08981984],
       [ -33.01764348,  -24.17107725]])

And how the chromaticities depend on the sextupole strengths:

In [6]:
v, J = sb.value_and_jacobian([tw.dq1, tw.dq2], [sb.Knob(sf, "Kn2"), sb.Knob(sd, "Kn2")])
J  # [[ddq1/dksf, ddq1/dksd], [ddq2/dksf, ddq2/dksd]]

array([[ 0.38629473,  0.27117375],
       [-0.31017778, -0.34945196]])

No truncation order has to be chosen: each quantity knows the order it needs, here the second
order in δ for the chromaticities.

Any parameter that can be set with `object.parameter = value` can be a knob: an element's
strength, length or misalignment, or a variable of [context](defexpr.md) (see
[Knobs](#optimize.knobs)).

`value` computes only the values. `deriv(quantity, knob)` is the derivative as a new quantity,
so that derivatives can be taken again. For example, the second derivative of the horizontal
tune with respect to the strength of `qf`:

In [7]:
sb.value(sb.deriv(sb.deriv(tw.q1, kf), kf))  # -5.5676

-5.567590565172136

Or how the first δ-derivative of the resonance driving term $h_{2000}$ at the beginning depends
on the strength of `sf` (the `h` terms are complex numbers):

In [8]:
sb.value(sb.deriv(tw.dh2000[1], sb.Knob(sf, "Kn2")))  # 1.209 - 0.1048j

(1.209004212165644-0.1047796752464366j)

These expressions are also what is matched in [Matching and Optimization](optimize.md).

## Taylor series in the parameters with GTPSA

The derivatives above are computed by making the knobs parameters of the truncated power series
that `twiss` computes with, and restoring the lattice afterwards. This can also be done by hand,
to get the complete Taylor series in the parameters. It uses the truncated power series package
[GTPSA](https://github.com/bmad-sim/GTPSA.jl), whose
[quickstart guide](https://bmad-sim.github.io/GTPSA.jl/stable/quickstart/) explains the ideas.
The normal forms themselves are computed by the
[NonlinearNormalForm](https://github.com/bmad-sim/NonlinearNormalForm.jl) package.

First, a GTPSA `Descriptor` defines the variables, the parameters, and their truncation orders.
Here, it has the six phase space variables to order 1, two parameters (the changes of the two
quadrupole strengths) to order 1, and a maximum total order of 2:

In [9]:
dnf = sb.Descriptor([1, 1, 1, 1, 1, 1], 2, [1, 1], 1)
dnf

Descriptor(NV=6, MO=2, NP=2, PO=1, NO=[ 1 1  1 1  1 1])

The tunes and the beta functions come from the first-order terms of the map in the phase space
variables. Their first-order dependence on a parameter is therefore in terms of total order 2:
one variable times one parameter. Higher orders work too, but take longer to compute.

The parameters are then added to the strengths:

In [10]:
dk = sb.params(dnf)
qf.Kn1 = 0.36 + dk[0]
qd.Kn1 = -0.36 + dk[1]

and `twiss` is given the Descriptor with `GTPSA_descriptor`:

In [11]:
twp = sb.twiss(fodo, GTPSA_descriptor=dnf)
twp

Twiss:
summ:
 q1     = 0.05344930466011301 + 0.08756175774882742 δ + 0.553969060487466 Δk₁ + 0.4091622849161083 Δk₂
 q2     = 0.05253999423230937 - 0.004310546648378302 δ - 0.4162809793354111 Δk₁ - 0.5635546864898774 Δk₂
 etac   = 0.02021231039014999 - 0.4191932585716806 Δk₁ - 0.3103664171777638 Δk₂
 alphac = 0.020212311196075756 - 0.41919325857168055 Δk₁ - 0.31036641717776375 Δk₂

df:
9×26 DataFrame
 Row │ index  name  kind        s    beta1                              phi1                       ⋯
     │                          [m]  [m]                                [2π]                       ⋯
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │     1  qf    Quadrupole  0.0  TPS64{Dynamic}:\nDescriptor(NV=6…  TPS64{Dynamic}:\nDescripto ⋯
   2 │     2  sf    Sextupole   0.5  TPS64{Dynamic}:\nDescriptor(NV=6…  TPS64{Dynamic}:\nDescripto
   3 │     3  b     SBend       0.7  TPS64{Dynamic}:\nDescriptor(NV=6…  TPS64{Dynamic}:\nDesc

With parameters in the Descriptor, `twiss` returns its results as Taylor series
(`as_taylor_series=True`). The printed tunes now include their dependence on the strengths of
`qf` and `qd`. `grad` gives the first derivatives with respect to the parameters as a vector:

In [12]:
sb.grad(twp.q1)  # [dq1/dkf, dq1/dkd], as above

array([0.55396906, 0.40916228])

In [13]:
sb.grad(twp.beta1[qd])  # [dbeta1/dkf, dbeta1/dkd]

array([-110.1325315 ,  -78.08981984])

Afterwards, the strengths of `qf` and `qd` are still Taylor series, even if numbers are set back
into them. Before using the beamline again, call {jl}`scalarize!`{py}`sb.scalarize`. It replaces
every Taylor series
in the beamline by its value (its 0th order part):

In [14]:
sb.scalarize(fodo)

Beamline:
 species_ref = electron
 pc_ref = 1.8e10

  Index   Name   Kind         L [m]   s [m]                s_downstream [m]
  1       qf     Quadrupole   0.5     0.0f0                0.5
  2       sf     Sextupole    0.2     0.5                  0.7
  3       b      SBend        1.2     0.7                  1.9
  4       d      Drift        0.1     1.9                  2.0
  5       qd     Quadrupole   0.5     2.0                  2.5
  6       sd     Sextupole    0.2     2.5                  2.7
  7       b      SBend        1.2     2.7                  3.9000000000000004
  8       d      Drift        0.1     3.9000000000000004   4.0


:::{warning}
Mixing Taylor series of two different Descriptors aborts the Julia process, with an error of
the GTPSA C library. For example, `twiss(fodo)` above, before {jl}`scalarize!`{py}`scalarize`, would
abort Julia{pytext}` (and Python with it)`,
because `twiss` uses its own Descriptor.
:::